# B92 Quantum Key Distribution Protocol
## Implementation and Security Analysis with Qiskit

This notebook presents a complete implementation of the B92 protocol.

### Document Structure

**Overview** - Introduction to the B92 protocol

**Part 1: Protocol Implementation** - Step-by-step implementation
   - Alice's preparation
   - Bob's measurement

**Part 2: Complete Protocol Execution** - Demonstration without attacks
   - Secure key exchange results
   
**Part 3: Security Analysis** - Why B92 is secure

In [18]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeVigoV2
import numpy as np

## Overview

The B92 protocol is a "prepare and measure" Quantum Key Distribution (QKD) protocol created by Charles Bennett in 1992. It is a simplified version of the BB84 where just two non-orthogonal states are used instead of four.

We are not going to cover the theoretical foundation of this protocol since it is the same that the BB84 one, so please refer to that notebook.

## Part 1: Protocol Implementation
### 1.1 Alice's Step - Bit Generation and Preparation

The first step is to generate the bits that will be sent. This is done classically using NumPy.

In [19]:
def alice_bit_generation(n_bits):
    alice_bits= np.random.randint(0,2,size=n_bits,dtype='int8') 
    return alice_bits       

#### Alice's Qubit Preparation

The next step is to convert bits into qubits. In this protocol the mapping is the following:
$$
0 \rightarrow |0 \rangle \quad ; \quad 1 \rightarrow |+ \rangle= \frac{1}{\sqrt{2}}\left( |0 \rangle + |1 \rangle \right)
$$

 Since Qiskit works with quantum circuits, we represent individual qubits as single-qubit circuits. The encoding is done with the following code:
- Bit value determines the H gate (1 → apply H gate)

In [20]:
def alice_prepare_qubits(alice_bits):
    n_bits=len(alice_bits)      

    qubits=[] #Empty list to store the qubits

    for i in range(n_bits):

        qc=QuantumCircuit(1) #We initialize the qubit, by default at |0>

        if alice_bits[i]: #This executes if the classical bit is a 1
            qc.h(qubit=0) #Add an H gate,this changes |0> to |+>

        qc.barrier() #We add a barrier to represent the sending

        qubits.append(qc)
        
    return qubits


### 1.2 Bob's Step - Measurement and processing

#### Bob's measurement 

Once the qubits arrive to Bob, he randomly decides basis and measure the states. We implement Bob's measurement using Qiskit's simulator with optional noise modeling.

In [21]:
#Qiskit have different templates to noise model, max number of qubits of a circuit, available gates...
#This are fake providers
def bob_measures(qubits,noise_model=None):

    n_bits=len(qubits)
    
    bob_basis=np.random.randint(0,2,size=n_bits,dtype='int8') #Bob generates his basis 0=Z, 1=X

    for i in range(n_bits):
        if bob_basis[i]: #If Bob measures in X basis we have to add an H
            qubits[i].h(0) 
        qubits[i].measure_all() #Instruction to measure the qubits of a circuit

    backend=FakeVigoV2() #We choose our template, other fake providers exist
    simulator=AerSimulator.from_backend(backend) #This is the actual simulator initialized from our backend
    qubits_t=transpile(qubits,backend) #This adapts the circuit to the chosen backend

    #We execute the simulation, Qiskit works with lists and arrays by default

    if noise_model:  #noise_model= True if we want to simulate noise
        job=simulator.run(qubits_t,shots=1)
    else:
        job=simulator.run(qubits_t,noise_model=None,shots=1)

    #We extract the results
    results=job.result()

    #We extract the value of the count
    counts=results.get_counts() #This is a dictionary of how many 0's and 1's

    #We convert from the list of dictionaries to a bitstring (array of 0's and 1's)
    bob_meas = np.array([int(list(out.keys())[0]) for out in counts])
    return bob_meas,bob_basis

#### Bob's Post-Processing

The measurement of Bob are not directly the bits that Alice sent, to understand why let's see all the possibilities:

1. **Bob measured 0** 

    a) **He measured in the Z basis**
      - It is the guaranteed outcome if Alice sent $|0 \rangle$ and a 50% possibility outcome if Alice sent $|+ \rangle$. We cannot conclude anything from this .
   
   b) **He measured in the X basis**

      - It is the guaranteed outcome if Alice sent $|+ \rangle$ and a 50% possibility outcome if Alice sent $|0 \rangle $. We cannot conclude anything from this .
   
2. **Bob measured 1**

   a) **He measured in the Z basis**
      - It is an impossible outcome if Alice sent $|0 \rangle$ and a 50% possibility outcome if Alice sent $|+ \rangle$. We can conclude that Alice sent $|+ \rangle$ and therefore her bit is a 1.
   
   b) **He measured in the X basis**
      - It is an impossible outcome if Alice sent $|+ \rangle$ and a 50% possibility outcome if Alice sent $|0 \rangle$. We can conclude that Alice sent $|0 \rangle$ and therefore her bit is a 0.

So Bob has to post-process and only keep those bits which are useful to recover Alice's, This strategy implies that only 25% of the sent qubits will survive to this step. We implement this processing with NumPy arrays.

In [22]:
def bob_post_processing(bob_meas,bob_basis):
    
    #Boolean mask that tell us if bob measured a 1

    useful_indexes= bob_meas==1

    #We now extract which basis we used in those cases

    useful_basis= bob_basis[useful_indexes]

    # If outcome 1 and basis Z(0) --> Alice qubit was a |+> --> Alice bit was a 1
    # If outcome 1 and basis X(1) --> Alice qubit was a |1> --> Alice bit was a 0
    #This can be simply implemented as performing a NOT to the basis of the useful cases

    filtered_bob_bits=1-useful_basis #base Z(0) -> bit 1, base X(1) -> bit 0

    return filtered_bob_bits,useful_indexes

### 1.3 Classical Communication - Filtering and QBER calculation

#### Two Essential Steps

The next steps involve classical communication through an authenticated channel and consist of two critical operations:
1. **Filter sharing**: Bob sends Alice the indexes of the qubits that yielded a useful outcome.
2. **QBER Calculation**: Detect eavesdropping by calculating the Quantum Bit Error Rate.

#### Step 1: Filter sharing

Filter sharing consists of filtering Alice's bit to only the ones that gave Bob information using their indexes. Once Bob receives all qubits, he publicly shares this indexes with Alice. This is easily implemented using NumPy arrays.

In [23]:
def public_filtering(alice_bits,useful_indexes):

    #We simply apply the same boolean mask as before
    filtered_alice_bits=alice_bits[useful_indexes]

    return filtered_alice_bits

#### Step 2: QBER Calculation and Eavesdropping Detection

After filtering, Alice and Bob publicly reveal a percentage of the bits (typically 50%) to calculate the QBER metric. This metric measures the percentage of bits that differ after base sifting; ideally, it should be 0. However, this metric provides an estimate of transmission noise and serves as an eavesdropper detector. Again, we implement this using NumPy arrays.

In [24]:
def public_qber_calculation(filtered_alice_bits,filtered_bob_bits,frac=0.5):
    
    n_sifted_bits = len(filtered_alice_bits)
    n_public_bits = int(n_sifted_bits * frac) #Number of qubits that will be made public

    #We choose randomly the index of the public bits
    public_indexes = np.random.choice(n_sifted_bits,size=n_public_bits,replace=False)

    #Again, we make use of a boolean mask
    public_mask = np.zeros(n_sifted_bits, dtype=bool)
    public_mask[public_indexes] = True



    #We extract the public bits from the mask
    alice_public = filtered_alice_bits[public_mask]
    bob_public = filtered_bob_bits[public_mask]

    # We calculate the errors and the qubits using the public bits
    bit_errors = np.sum(alice_public != bob_public) 
    qber = bit_errors / n_public_bits


    #The remaining bits are the key
    private_mask=~public_mask #private mask= NOT public mask

    remaining_alice_bits = filtered_alice_bits[private_mask]
    remaining_bob_bits = filtered_bob_bits[private_mask]
    return qber,remaining_alice_bits,remaining_bob_bits


## Part 2: Complete Protocol Execution - Without Eavesdropping

### 2.1 Demonstration: Secure Key Exchange

The remaining bits after sifting and testing are the extracted key. Since they are still secret and in an ideal case (no noise, no attack), they would have the same value for both Alice and Bob. Let's execute the complete protocol and verify the results:

In [25]:
n_bits=1024
noise_model=None

#Alice part
alice_bits=alice_bit_generation(n_bits=n_bits)
qubits=alice_prepare_qubits(alice_bits=alice_bits)

#Bob part 
bob_meas,bob_basis=bob_measures(qubits=qubits,noise_model=noise_model)
filtered_bob_bits,useful_indexes=bob_post_processing(bob_meas=bob_meas,bob_basis=bob_basis)

#Classical public authenticated communication

filtered_alice_bits=public_filtering(alice_bits=alice_bits,useful_indexes=useful_indexes)
qber,alice_key,bob_key=public_qber_calculation(filtered_alice_bits=filtered_alice_bits,
                                                  filtered_bob_bits=filtered_bob_bits)

#Final result presentation

print(f"The QBER is {qber}")
print("Alice's key is: ")
print(alice_key)
print("Bob's key is")
print(bob_key)
print("Are the same key? ",np.array_equal(alice_key,bob_key))

The QBER is 0.0
Alice's key is: 
[1 0 0 0 1 1 1 0 0 1 0 1 0 0 0 0 0 0 1 1 1 1 0 0 1 1 1 1 0 0 0 0 0 1 0 1 0
 1 1 0 0 0 1 0 0 0 0 1 1 1 0 1 1 0 0 1 1 0 0 0 1 1 1 0 1 1 0 1 1 0 1 0 1 0
 0 1 0 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1
 0 1 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 1 0 0 1 1]
Bob's key is
[1 0 0 0 1 1 1 0 0 1 0 1 0 0 0 0 0 0 1 1 1 1 0 0 1 1 1 1 0 0 0 0 0 1 0 1 0
 1 1 0 0 0 1 0 0 0 0 1 1 1 0 1 1 0 0 1 1 0 0 0 1 1 1 0 1 1 0 1 1 0 1 0 1 0
 0 1 0 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1
 0 1 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 1 0 0 1 1]
Are the same key?  True


### 2.2 Observations

If noise is present in the quantum channel, the keys will differ, requiring additional error reconciliation algorithms such as cascade or minnow to correct bit errors. These algorithms are beyond the scope of this notebook. It is also common to perform privacy amplification to achieve arbitrary privacy.

Now the key question arises: **Is this protocol secure against eavesdropping attacks?**

## Part 3: Security Analysis 

### Eve's Attack ###

The B92 protocol bases its theoretical security on the laws of quantum mechanics, specifically on the impossibility of deterministically distinguishing between non-orthogonal quantum states.

The strategy of the attacker Eve consists of intercepting the qubits sent by Alice and measuring them to obtain the bit value. Two fundamental constraints prevent her success: the no-cloning theorem prohibits copying the quantum state, and measuring non-orthogonal states irreversibly disturbs the original system.

In the B92 protocol, Alice transmits only two non-orthogonal states, such as $|0 \rangle$  ( bit 0) and $|+ \rangle$ (bit 1). Not knowing the sent state, Eve's strategy involves choosing a basis, measuring the qubit, and resending a new state to Bob based on her result. 

The security of the B92 protocol relies on the fact that any attempt to extract information from non-orthogonal quantum states introduces unavoidable perturbations. Any interception raises the error rate in the conclusive measurements (QBER) to statistically obvious levels, allowing Alice and Bob to conclusively identify the attack.